In [1]:
from graph import Graph
from breadth_first_paths import BreadthFirstPaths
from depth_first_paths import DepthFirstPaths
import pandas as pd

# BFS e DFS — Estados do Nordeste

Grafo com os 9 estados do Nordeste e suas fronteiras terrestres.

Mapeamento (ordem alfabética):
```
0: AL  1: BA  2: CE  3: MA  4: PB  5: PE  6: PI  7: RN  8: SE
```

<!-- ![Região Nordeste](../dados/mapa-da-regiao-nordeste.jpg) -->
![Região Nordeste](../dados/grafo-da-regiao-nordeste.png)

## Funções auxiliares e carregamento do grafo

In [2]:
states = ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE']

def path_to_states(path):
    """Converte lista de índices em lista de siglas de estados."""
    if path is None:
        return None
    return [states[i] for i in path]

def print_path(path):
    if path is None:
        print("Caminho não encontrado.")
    else:
        print(' -> '.join(path_to_states(path)))

def state(num):
    return states[num]

def num(s):
    return states.index(s)

In [3]:
def load_graph_from_file(file_path):
    with open(file_path, 'rt') as f:
        v = int(f.readline())
        e = int(f.readline())
        G = Graph(v)
        for line in f.readlines():
            if not line.startswith('#'):
                parts = line.split()
                if len(parts) == 2:
                    G.add_edge(int(parts[0]), int(parts[1]))
    return G

G = load_graph_from_file('../dados/nordeste.txt')

print(f"Número de vértices: {G.V}")
print(f"Número de arestas: {G.E}")

Número de vértices: 9
Número de arestas: 14


## Entrada: estado de origem e destino

Alterar as variáveis `ORIGEM` e `DESTINO` para testar diferentes pares.

In [4]:
ORIGEM  = 'CE'
DESTINO = 'SE'

src = num(ORIGEM)
dst = num(DESTINO)

# Instancia BFS e DFS a partir da origem
bfp = BreadthFirstPaths(G, source=src)
dfp = DepthFirstPaths(G, source=src)

print(f"Origem : {ORIGEM} (vértice {src})")
print(f"Destino: {DESTINO} (vértice {dst})")

Origem : CE (vértice 2)
Destino: SE (vértice 8)


## 1. É possível sair de X e chegar a Y atravessando apenas fronteiras terrestres?

In [5]:
alcancavel = bfp.has_path_to(dst)   # BFS e DFS têm o mesmo conjunto de vértices alcançáveis

if alcancavel:
    print(f"✅ Sim. É possível sair de {ORIGEM} e chegar a {DESTINO} por fronteiras terrestres.")
else:
    print(f"❌ Não. {DESTINO} não é alcançável a partir de {ORIGEM} por fronteiras terrestres.")

✅ Sim. É possível sair de CE e chegar a SE por fronteiras terrestres.


## 2. Caminho encontrado pela DFS de X até Y

In [6]:
dfs_path = dfp.path_to(dst)

print(f"Caminho DFS de {ORIGEM} até {DESTINO}:")
print_path(dfs_path)

Caminho DFS de CE até SE:
CE -> PB -> PE -> AL -> BA -> SE


## 3. Caminho encontrado pela BFS de X até Y

In [7]:
bfs_path = bfp.path_to(dst)

print(f"Caminho BFS de {ORIGEM} até {DESTINO}:")
print_path(bfs_path)

Caminho BFS de CE até SE:
CE -> PE -> AL -> SE


## 4. Estados alcançáveis a partir de X

In [8]:
reachable_indices = dfp.reachable()
reachable_states  = [states[i] for i in reachable_indices]

print(f"Estados alcançáveis a partir de {ORIGEM}:")
print(', '.join(reachable_states))

Estados alcançáveis a partir de CE:
AL, BA, CE, MA, PB, PE, PI, RN, SE


## 5. Ordem de visita da DFS a partir de X

In [9]:
dfs_order_states = [states[v] for v in dfp.order]

print(f"Ordem de visita da DFS a partir de {ORIGEM}:")
for i, s in enumerate(dfs_order_states, start=1):
    print(f"  {i}º: {s}")

if dfs_path:
    print()
    print(f"Posições de visita no caminho DFS até {DESTINO}: {dfp.visit_order_to(dst)}")

Ordem de visita da DFS a partir de CE:
  1º: CE
  2º: PB
  3º: PE
  4º: AL
  5º: BA
  6º: PI
  7º: MA
  8º: SE
  9º: RN

Posições de visita no caminho DFS até SE: [1, 2, 3, 4, 5, 8]


## 6. Ordem de visita da BFS a partir de X

In [10]:
bfs_order_states = [states[v] for v in bfp.order]

print(f"Ordem de visita da BFS a partir de {ORIGEM}:")
for i, s in enumerate(bfs_order_states, start=1):
    print(f"  {i}º: {s}")

if bfs_path:
    print()
    print(f"Posições de visita no caminho BFS até {DESTINO}: {bfp.visit_order_to(dst)}")

Ordem de visita da BFS a partir de CE:
  1º: CE
  2º: PB
  3º: PE
  4º: PI
  5º: RN
  6º: AL
  7º: BA
  8º: MA
  9º: SE

Posições de visita no caminho BFS até SE: [1, 3, 6, 9]


## Resumo completo

In [11]:
print("=" * 50)
print(f"  Origem: {ORIGEM}   |   Destino: {DESTINO}")
print("=" * 50)

print(f"\n[1] Alcançável: {'Sim' if alcancavel else 'Não'}")

print(f"\n[2] Caminho DFS: ", end="")
print_path(dfs_path)

print(f"[3] Caminho BFS: ", end="")
print_path(bfs_path)

print(f"\n[4] Estados alcançáveis a partir de {ORIGEM}:")
print(f"    {', '.join(reachable_states)}")

print(f"\n[5] Ordem de visita DFS: {' -> '.join(dfs_order_states)}")
print(f"[6] Ordem de visita BFS: {' -> '.join(bfs_order_states)}")

print()
df = pd.DataFrame({
    'Estado': states,
    'Alcançável': [dfp.has_path_to(num(s)) for s in states],
    'Caminho DFS': [path_to_states(dfp.path_to(num(s))) for s in states],
    'Caminho BFS': [path_to_states(bfp.path_to(num(s))) for s in states],
})
print(df.to_string(index=False))

  Origem: CE   |   Destino: SE

[1] Alcançável: Sim

[2] Caminho DFS: CE -> PB -> PE -> AL -> BA -> SE
[3] Caminho BFS: CE -> PE -> AL -> SE

[4] Estados alcançáveis a partir de CE:
    AL, BA, CE, MA, PB, PE, PI, RN, SE

[5] Ordem de visita DFS: CE -> PB -> PE -> AL -> BA -> PI -> MA -> SE -> RN
[6] Ordem de visita BFS: CE -> PB -> PE -> PI -> RN -> AL -> BA -> MA -> SE

Estado  Alcançável                  Caminho DFS      Caminho BFS
    AL        True             [CE, PB, PE, AL]     [CE, PE, AL]
    BA        True         [CE, PB, PE, AL, BA]     [CE, PE, BA]
    CE        True                         [CE]             [CE]
    MA        True [CE, PB, PE, AL, BA, PI, MA]     [CE, PI, MA]
    PB        True                     [CE, PB]         [CE, PB]
    PE        True                 [CE, PB, PE]         [CE, PE]
    PI        True     [CE, PB, PE, AL, BA, PI]         [CE, PI]
    RN        True                 [CE, PB, RN]         [CE, RN]
    SE        True     [CE, PB, PE, AL, 